# Main model comparison — performance and paired significance

Collect test-set bootstrap metrics and BH-adjusted paired MCC permutation comparisons, and mark significant M3Care/FuseMoE rows.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import os
print(os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Update this path to match your environment.
ROOT = Path("/data/logs/ablation/model_comparison")
OUTPUT = Path("/data/outputs/report/model_comparison/model_performance_summary.csv")

## 1. Extract and format test metrics

`point` is the original test metric estimate; `ci_lower` and `ci_upper` are the bootstrap confidence interval bounds. Only rows with `split == "test"` are retained.

In [ ]:
METRICS = ['auroc', 'mcc', 'recall', 'specificity', 'ba']
MODEL_NAMES = {'concat': 'Concatenation', 'smp': 'FuseMoE', 'fusemoe': 'FuseMoE', 'm3care': 'M3Care'}


def collect_performance(root: Path) -> pd.DataFrame:
    records = []
    files = sorted(root.rglob('bootstrap_metrics.csv'))
    if not files:
        raise FileNotFoundError(f'No bootstrap_metrics.csv files found under {root}')

    for file in files:
        relative = file.relative_to(root)
        parts = relative.parts
        if not parts or parts[0] not in {'deterioration', 'hospitalisation'}:
            continue
        pathway = parts[0]
        window = next((p for p in parts if p in {'currWindow', 'dualWindow'}), '-')
        df = pd.read_csv(file)
        required = {'split', 'metric', 'point', 'ci_lower', 'ci_upper', 'outcome', 'model'}
        if not required.issubset(df.columns):
            print(f'SKIP {file}: missing {sorted(required - set(df.columns))}')
            continue
        df = df.loc[(df['split'].astype(str).str.lower() == 'test') & df['metric'].isin(METRICS)].copy()
        if df.empty:
            print(f'SKIP {file}: no test metrics')
            continue
        for _, row in df.iterrows():
            records.append({
                'Pathway': pathway.title(),
                'Window': window,
                'Outcome': str(row['outcome']),
                'Model': MODEL_NAMES.get(str(row['model']).lower(), str(row['model'])),
                'Run ID': str(row.get('run_id', file.parent.name)),
                'Metric': str(row['metric']),
                'Point': row['point'],
                'CI lower': row['ci_lower'],
                'CI upper': row['ci_upper'],
                'Source': str(file),
            })
    if not records:
        raise ValueError('No matching test metrics found.')
    long = pd.DataFrame(records)
    keys = ['Pathway', 'Window', 'Outcome', 'Model', 'Run ID']
    long['Display'] = long.apply(
        lambda r: f"{r['Point']:.3f} ({r['CI lower']:.3f}-{r['CI upper']:.3f})"
        if pd.notna(r['Point']) and pd.notna(r['CI lower']) and pd.notna(r['CI upper'])
        else 'NA', axis=1)
    wide = long.pivot_table(index=keys, columns='Metric', values='Display', aggfunc='first').reset_index()
    wide.columns.name = None
    wide = wide.rename(columns={'auroc': 'AUROC (95% CI)', 'mcc': 'MCC (95% CI)',
        'recall': 'Recall (95% CI)', 'specificity': 'Specificity (95% CI)', 'ba': 'BA (95% CI)'})
    columns = keys + ['AUROC (95% CI)', 'MCC (95% CI)', 'Recall (95% CI)', 'Specificity (95% CI)', 'BA (95% CI)']
    wide = wide.reindex(columns=columns)
    return wide.sort_values(keys).reset_index(drop=True)



## 2. Collect all model results

In [ ]:
summary = collect_performance(ROOT)
print(f"Found {len(summary)} model/outcome rows")
display(summary)

## 3. Explore results by pathway and observation window

In [ ]:
display(summary.loc[summary["Pathway"] == "Deterioration"])

display(summary.loc[(summary["Pathway"] == "Hospitalisation") & (summary["Window"] == "currWindow")])

display(summary.loc[(summary["Pathway"] == "Hospitalisation") & (summary["Window"] == "dualWindow")])

## 4. Check for duplicate model/outcome combinations

If duplicates appear, inspect the `Run ID` and source files before selecting the runs to report.

In [ ]:
duplicates = (summary.groupby(["Pathway", "Window", "Outcome", "Model"], dropna=False)
              .size().reset_index(name="n_runs").query("n_runs > 1"))
if duplicates.empty:
    print("No duplicate model/outcome combinations found.")
else:
    display(duplicates)

## 5. Export the summary

In [ ]:
summary.to_csv(OUTPUT, index=False)
print(f"Saved {len(summary)} rows to {OUTPUT.resolve()}")

## 6. Load and combine BH-adjusted paired MCC tests

Point `BH_ROOT` to the **separate** significance-results tree. Files are matched by pathway, window and outcome (from their folder paths), and by the tested model. Only `concat`-versus-`m3care` and `concat`-versus-`smp` comparisons are used for the performance-table stars. A star denotes `reject_h0 == True` after BH correction; it **does not indicate which model performs better**.


In [ ]:
BH_ROOT = Path("/data/outputs/model_comparison") 

def collect_bh_tests(root: Path) -> pd.DataFrame:
    records = []
    for file in sorted(root.rglob("mcc_permutation_bh_test.csv")):
        parts = file.relative_to(root).parts
        if not parts or parts[0] not in {"deterioration", "hospitalisation"}:
            continue
        pathway = parts[0].title()
        window = next((x for x in parts if x in {"currWindow", "dualWindow"}), "—")
        # BH tree: deterioration/<short_outcome>/file or
        # hospitalisation/<window>/<short_outcome>/file.
        outcome_folder = file.parent.name
        data = pd.read_csv(file)
        required = {"model_a", "model_b", "delta_mcc", "p_value", "p_value_bh", "reject_h0"}
        if not required.issubset(data.columns):
            raise ValueError(f"Missing columns in {file}: {sorted(required - set(data.columns))}")
        for _, r in data.iterrows():
            raw = r["reject_h0"]
            flag = str(raw).strip().lower()
            if flag not in {"true", "false", "1", "0"}:
                raise ValueError(f"Unexpected reject_h0={raw!r} in {file}")
            records.append({"Pathway": pathway, "Window": window,
                            "Outcome folder": outcome_folder,
                            "model_a": str(r["model_a"]).lower().strip(),
                            "model_b": str(r["model_b"]).lower().strip(),
                            "delta_mcc": r["delta_mcc"], "p_value": r["p_value"],
                            "p_value_bh": r["p_value_bh"],
                            "reject_h0": flag in {"true", "1"},
                            "Source": str(file)})
    if not records:
        raise FileNotFoundError(f"No mcc_permutation_bh_test.csv found under {root}")
    return pd.DataFrame(records)

bh_tests = collect_bh_tests(BH_ROOT)
display(bh_tests)


## 7. Match significant comparisons to performance rows

The significance folder uses shortened outcome names (e.g. `cardiac_arrest`), while performance CSVs may use `deterioration_cardiac_arrest`. The matching function strips the pathway prefix and preserves the observation-window grouping. Duplicated or unmatched comparisons are reported instead of silently selecting a result.


In [ ]:
def normalise_outcome(value):
    value = str(value).strip().lower()
    for prefix in ("deterioration_", "hospitalisation_"):
        if value.startswith(prefix):
            return value[len(prefix):]
    return value

def normalise_model(value):
    value = str(value).strip().lower()
    aliases = {
        "smp": "fusemoe",
        "fusemoe": "fusemoe",
        "m3care": "m3care",
        "concat": "concat",
        "concatenation": "concat",
    }
    return aliases.get(value, value)

def normalise_window(value):
    value = str(value).strip().lower()
    if value in ("-", "—", "", "none", "nan"):
        return "none"
    return value

# df: performance summary from the previous notebook cells
# bh_df: merged BH comparison results

perf = summary.copy()
bh = bh_tests.copy()

# Normalise performance keys
perf["_outcome"] = perf["Outcome"].map(normalise_outcome)
perf["_model"] = perf["Model"].map(normalise_model)
perf["_window"] = perf["Window"].map(normalise_window)
perf["_pathway"] = perf["Pathway"].str.strip().str.lower()

# Normalise BH keys
bh["_outcome"] = bh["Outcome folder"].map(normalise_outcome)
bh["_model"] = bh["model_b"].map(normalise_model)
bh["_window"] = bh["Window"].map(normalise_window)
bh["_pathway"] = bh["Pathway"].str.strip().str.lower()

keys = ["_pathway", "_window", "_outcome", "_model"]

# Ensure there is at most one BH result per model/outcome
duplicates = bh[bh.duplicated(keys, keep=False)]
if not duplicates.empty:
    display(duplicates)
    raise ValueError("Duplicate BH comparisons detected.")

# Keep the significance result and available statistics
bh_columns = keys + [
    c for c in ["reject_h0", "p_value_bh", "delta_mcc"]
    if c in bh.columns
]

summary = perf.merge(
    bh[bh_columns],
    on=keys,
    how="left",
    validate="many_to_one",
    indicator=True,
)

# Check matching coverage before adding significance markers
unmatched = summary[
    (summary["_model"].isin(["m3care", "fusemoe"]))
    & (summary["_merge"] == "left_only")
]

if not unmatched.empty:
    print("WARNING: Unmatched performance rows:")
    display(unmatched[["Pathway", "Window", "Outcome", "Model"]])
else:
    print("All M3Care and FuseMoE rows matched successfully.")

# Robust conversion of reject_h0 to boolean
def is_significant(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() in ("true", "1", "yes")

summary["Significant"] = summary["reject_h0"].map(is_significant)

# Add * only to significant FuseMoE / M3Care rows
summary["Model"] = summary["Model"] + summary["Significant"].map(
    {True: "*", False: ""}
)

# Remove temporary merge keys
summary = summary.drop(
    columns=keys + ["_merge"]
)

display(summary)


## 8. Export BH comparisons and starred performance table

`*` means a BH-adjusted paired MCC permutation comparison against concatenation rejected the null hypothesis. Consult `delta_mcc` to determine the direction of the difference.


In [ ]:
OUTPUT_BH = Path("/data/outputs/report/model_comparison/mcc_bh_comparisons_merged.csv")
OUTPUT_SUMMARY_BH = Path("/data/outputs/report/model_comparison/model_performance_summary_with_significance.csv")

In [ ]:
bh_tests.to_csv(OUTPUT_BH, index=False)
summary.to_csv(OUTPUT_SUMMARY_BH, index=False)


# M3Care ablation results — explicitly selected run folders

Specify **exact run directories** rather than recursively scanning a results tree. Each directory must contain `bootstrap_metrics.csv`. Give each variant a unique label (e.g. `No GCN imputation`, `Masked fusion`) and a group (`Deterioration` / `Hospitalisation`, plus optional window). This section does **not** apply the main-comparison BH stars: ablation significance requires separate matched tests.

The output uses the same five metrics and 95% bootstrap CIs as the main-comparison table. It keeps both numeric point estimates and formatted values, so you can later highlight the best variant. 

## 1. Model architecture related ablations

In [ ]:
# Replace these examples with your actual run folders.
# You can use an absolute folder path or the bootstrap_metrics.csv file itself.
ABLATION_RUNS = [
    # Deterioration
    {
       "label": "Baseline", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_base_imputation_maskFusion_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "Baseline"
    },
    {
        "label": "Without GCN-based imputation", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_noImputation_maskFusion_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "GCN imputation"
    },
    {
        "label": "Without masked fusion", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_base_imputation_noMaskFusion_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "Fusion"
    },
    {
        "label": "Missingness embedding", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_missEmbed_maskFusion_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "Fusion"
    },
    {
        "label": "Imputed-for-missingness refinement", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_imputedMiss_maskFusion_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "GCN imputation"
    },
    {
        "label": "Cosine similarity-based graph stability loss (w = 0.01)", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_stabloss_cosine_w0_01_imputedMiss_maskFusion_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "GCN imputation"
    },
    {
        "label": "Gaussian similarity-based graph stability loss (w = 0.01)", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_stabloss_similarity_w0_01_imputedMiss_maskFusion_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "GCN imputation"
    },
    # Hospitalisation
    {
       "label": "Baseline", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_bl_imputation_maskFusion_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "Baseline"
    },
    {
        "label": "Without GCN-based imputation", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_noImputation_maskFusion_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "GCN imputation"
    },
    {
        "label": "Without masked fusion", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_imputation_noMaskFusion_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "Fusion"
    },
    {
        "label": "Missingness embedding", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_missEmbed_maskFusion_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "Fusion"
    },
    {
        "label": "Imputed-for-missingness refinement", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_imputedMiss_maskFusion_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "GCN imputation"
    },
    {
        "label": "Cosine similarity-based graph stability loss (w = 0.01)", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_stabloss_cosine_w0_01_imputedMiss_maskFusion_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "GCN imputation"
    },
    {
        "label": "Gaussian similarity-based graph stability loss (w = 0.01)", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_stabloss_similarity_w0_01_imputedMiss_maskFusion_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "GCN imputation"
    },
]


ABLATION_METRICS = ["mcc", "auroc", "ba", "recall", "specificity"]

def load_selected_ablation_runs(run_specs):
    if not run_specs:
        raise ValueError("Fill ABLATION_RUNS with at least one run directory.")
    records = []
    for spec in run_specs:
        path = Path(spec["path"]).expanduser()
        csv_path = path if path.name == "bootstrap_metrics.csv" else path / "bootstrap_metrics.csv"
        if not csv_path.is_file():
            raise FileNotFoundError(f"Missing file: {csv_path}")
        data = pd.read_csv(csv_path)
        required = {"split", "metric", "point", "ci_lower", "ci_upper", "outcome"}
        missing = required - set(data.columns)
        if missing:
            raise ValueError(f"{csv_path} missing columns: {sorted(missing)}")
        selected = data.loc[
            data["split"].astype(str).str.lower().eq("test") &
            data["metric"].astype(str).str.lower().isin(ABLATION_METRICS)
        ].copy()
        if selected.empty:
            raise ValueError(f"No test metrics in {csv_path}")
        selected["metric"] = selected["metric"].str.lower()
        for _, r in selected.iterrows():
            records.append({
                "Pathway": spec["pathway"],
                "Window": spec.get("window", "-"),
                "Outcome": str(r["outcome"]),
                "Category": spec.get("category", "Other"),
                "Variant": spec["label"],
                "Metric": r["metric"],
                "Point": float(r["point"]),
                "CI lower": float(r["ci_lower"]),
                "CI upper": float(r["ci_upper"]),
                "Source": str(csv_path),
            })
    long = pd.DataFrame(records)
    key = ["Pathway", "Window", "Outcome", "Category", "Variant"]
    dup = long.duplicated(key + ["Metric"], keep=False)
    if dup.any():
        display(long.loc[dup].sort_values(key + ["Metric"]))
        raise ValueError("Duplicate metric rows for the same variant. Use unique labels or remove duplicate runs.")
    counts = long.groupby(key, dropna=False)["Metric"].nunique()
    if (counts != len(ABLATION_METRICS)).any():
        print("WARNING: Some runs do not have all five metrics:")
        display(counts[counts != len(ABLATION_METRICS)])
    long["Formatted"] = long.apply(
        lambda r: f'{r["Point"]:.3f} [{r["CI lower"]:.3f}-{r["CI upper"]:.3f}]', axis=1)
    formatted = long.pivot(index=key, columns="Metric", values="Formatted").reset_index()
    formatted.columns.name = None
    formatted = formatted.rename(columns={m: f"{m.upper()} (95% CI)" for m in ABLATION_METRICS})
    numeric = long.pivot(index=key, columns="Metric", values="Point").reset_index()
    numeric.columns.name = None
    return long, formatted, numeric


In [ ]:
if ABLATION_RUNS:
    ablation_long, ablation_table, ablation_points = load_selected_ablation_runs(ABLATION_RUNS)
    display(ablation_table)
else:
    print("Add your selected run folders to ABLATION_RUNS, then rerun this cell.")


#### Compare selected variants and export

`ablation_table` is formatted for reading and eventual LaTeX conversion. `ablation_points` contains the original numeric test point estimates, useful for ranking. Group by outcome and ablation category before comparing variants; no statistical significance is implied by the highest point estimate.


In [ ]:
if ABLATION_RUNS:
    display(ablation_points.sort_values(["Pathway", "Window", "Outcome", "Category", "mcc"], ascending=[True, True, True, True, False]))
    ablation_table.to_csv("/data/outputs/report/model_comparison/m3care_architecture_ablation_performance_summary.csv", index=False)
    ablation_long.to_csv("/data/outputs/report/model_comparison/m3care_architecture_ablation_performance_long.csv", index=False)
    ablation_points.to_csv("/data/outputs/report/model_comparison/m3care_architecture_ablation_performance_numeric.csv", index=False)
    print("Saved formatted, long-form and numeric ablation results.")


## 2. Contrastive learning

In [ ]:
CL_RUNS = [
    # Deterioration
    {
       "label": "Baseline", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_stabloss_similarity_w0_01_imputedMiss_maskFusion_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "Baseline"
    },
    {
        "label": "Full CL", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_deterBest_CL_finetune_bl_MV_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "CL_full"
    },
    {
        "label": "w/o missing-modality alignment", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_deterBest_CL_finetune_missW0_0_MV_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "CL_missingalign"
    },
    {
        "label": "w/o learned target", 
        "path": "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_deterBest_CL_finetune_targetContrastW0_0_MV_3_1ohe", 
        "pathway": "Deterioration", 
        "window": "—", 
        "category": "CL_learnedtarget"
    },
    # Hospitalisation
    {
       "label": "Baseline", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_imputedMiss_maskFusion_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "Baseline"
    },
    {
        "label": "Full CL", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_1S_missW0_5_EXT_ST_dualWin_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "CL_full"
    },
    {
        "label": "w/o missing-modality alignment", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_1S_missW0_0_EXT_ST_dualWin_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "CL_missingalign"
    },
    {
        "label": "w/o learned target", 
        "path": "/data/logs/ablation/m3care/hospitalisation/dualWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_1S_targetContrastW0_0_EXT_ST_dualWin_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "dualWindow", 
        "category": "CL_learnedtarget"
    },
    {
       "label": "Baseline", 
        "path": "/data/logs/ablation/m3care/hospitalisation/currWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_normloss_w0_01_imputedMiss_maskFusion_EXT_ST_currWin_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "currWindow", 
        "category": "Baseline"
    },
    {
        "label": "Full CL", 
        "path": "/data/logs/ablation/m3care/hospitalisation/currWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_finetune_missW0_5_EXT_ST_currWin_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "currWindow", 
        "category": "CL_full"
    },
    {
        "label": "w/o missing-modality alignment", 
        "path": "/data/logs/ablation/m3care/hospitalisation/currWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_finetune_missW0_0_EXT_ST_currWin_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "currWindow", 
        "category": "CL_missingalign"
    },
    {
        "label": "w/o learned target", 
        "path": "/data/logs/ablation/m3care/hospitalisation/currWindow/averageRecord/optionalHistory/normalised_ohe/hospitalisation_ext_stay_7/runs/m3care_m3care_hospBest_CL_finetune_targetContrastW0_0_EXT_ST_currWin_3_1hosp", 
        "pathway": "Hospitalisation", 
        "window": "currWindow", 
        "category": "CL_learnedtarget"
    },
]

In [ ]:
if CL_RUNS:
    cl_long, cl_table, cl_points = load_selected_ablation_runs(CL_RUNS)
    display(cl_table)

In [ ]:
if CL_RUNS:
    display(cl_points.sort_values(["Pathway", "Window", "Outcome", "Category", "mcc"], ascending=[True, True, True, True, False]))
    cl_table.to_csv("/data/outputs/report/model_comparison/m3care_cl_performance_summary.csv", index=False)
    cl_long.to_csv("/data/outputs/report/model_comparison/m3care_cl_performance_long.csv", index=False)
    cl_points.to_csv("/data/outputs/report/model_comparison/m3care_cl_performance_numeric.csv", index=False)
    print("Saved formatted, long-form and numeric CL results.")


## 3. Temporal module - mTAND

### Model performance comparison

In [ ]:
m3care_path = "/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_stabloss_similarity_w0_01_imputedMiss_maskFusion_3_1ohe"
fusemoe_path = "/data/logs/ablation/smp/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/smp_smp_0_333SharedWeight_0_1ModalityDropout_1e_1Lb_3_1ohe"
mtand_m3care_path = "/data/logs/deterioration/allStay/allRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/mtand_m3care_similarityloss_w0_01_imputedMiss_maskFusion_tm8_et16_3_1ohe"
mtand_fusemoe_path = "/data/logs/ablation/fusemoe/deterioration/allStay/allRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/fusemoe_fusemoe_0_333SharedWeight_0_1ModalityDropout_0_1EmbedDropout_1e_1Lb_mtandOn_3_1ohe"

# display name -> run directory; order fixes the bar order in every figure
model_paths = {
    "m3care": m3care_path,
    "m3care + mTAND": mtand_m3care_path,
    "fusemoe": fusemoe_path,
    "fusemoe + mTAND": mtand_fusemoe_path,
}

OUT_DIR = "/data/outputs/report"

In [ ]:
model_metrics = {
    name: pd.read_csv(os.path.join(path, "bootstrap_metrics.csv"))
    for name, path in model_paths.items()
}

# palette
palette = ["#6D2E8E", "#0E7C84", "#BE7C1C", "#3B4C61"]

In [ ]:
# test-set summary table: bootstrap point estimate and 95% CI per model and metric
metrics = ["mcc", "ba", "auroc", "specificity", "recall"]
metric_labels = {"auroc": "AUROC", "recall": "Recall", "specificity": "Specificity",
                 "mcc": "MCC", "ba": "BA"}

rows = {}
for name, df in model_metrics.items():
    test = df[df["split"] == "test"].set_index("metric")
    rows[name] = {
        metric_labels[m]: f"{test.loc[m, 'point']:.3f} ({test.loc[m, 'ci_lower']:.3f}-{test.loc[m, 'ci_upper']:.3f})"
        for m in metrics
    }
summary = pd.DataFrame(rows)
summary

In [ ]:
# best-vs-mTAND comparison: test-set performance across all five metrics (bootstrap mean and 95% CI)
# grouped by metric; colour encodes the fusion method, hatching marks the mTAND variant
models = list(model_paths)
model_colors = {
    "m3care": palette[1],
    "m3care + mTAND": palette[1],
    "fusemoe": palette[2],
    "fusemoe + mTAND": palette[2],
}
model_hatch = {m: ("//" if "mTAND" in m else "") for m in models}
# mTAND bars are drawn lighter (like the val bars in the Milestone II figures)
model_alpha = {m: (0.55 if "mTAND" in m else 1.0) for m in models}
split = "test"

x = np.arange(len(metrics))          # one group per metric
width = 0.8 / len(models)            # one bar per model within each group

fig, ax = plt.subplots(figsize=(8, 3.2), dpi=600)
# transparent background for both the figure and the axes
fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)

for mi, metric in enumerate(metrics):
    for ri, model in enumerate(models):
        df = model_metrics[model]
        offset = (ri - (len(models) - 1) / 2) * width
        row = df[(df["split"] == split) & (df["metric"] == metric)]
        p = row["point"].iloc[0]
        lo = p - row["ci_lower"].iloc[0]
        hi = row["ci_upper"].iloc[0] - p
        ax.bar(x[mi] + offset, p, width * 0.92,
               color=model_colors[model],
               alpha=model_alpha[model],
               hatch=model_hatch[model],
               edgecolor="white", linewidth=0.5,
               yerr=[[lo], [hi]], capsize=2,
               error_kw=dict(ecolor="lightgray", elinewidth=1))
        # value label above the error-bar cap
        ax.text(x[mi] + offset, p + hi + 0.012, f"{p:.3f}",
                ha="center", va="bottom", rotation=0,
                color="#404040", fontsize=6)

ax.set_xticks(x)
ax.set_xticklabels([metric_labels[m] for m in metrics], fontsize=9)
ax.set_xlim(-0.6, len(metrics) - 0.4)  # padding so the outer bars clear the spines
ax.tick_params(axis="x", length=0)       # drop x tick marks that poke into the bars
ax.set_ylabel("Performance")
ax.set_ylim(-0.025, 1.08)
ax.set_title("Best variant vs. mTAND variant: test-set performance")

# legend distinguishes the model; colour encodes the fusion method, hatching the mTAND variant
model_handles = [Patch(facecolor=model_colors[m], alpha=model_alpha[m],
                       hatch=model_hatch[m], edgecolor="white", label=m)
                 for m in models]
ax.legend(handles=model_handles, fontsize=8, loc="upper center",
          bbox_to_anchor=(0.5, -0.18), frameon=False, ncol=len(models))
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "results_best_vs_mtand_test_metrics.png"), dpi=600, transparent=True, bbox_inches="tight")
plt.show()

###  mTAND attention over observation time (from saved CSVs)

Reconstructs the `attn_marginal_over_time.png` calculation from `mtand_attention.py`: mean over reference-time rows, then normalise across observation-time bins. The CSVs are already averaged across eligible stays; the per-modality counts are not included in the CSV files.

In [ ]:
import re

In [ ]:
# Set this to the run directory containing mtand_attention/
# MTAND_RUN_DIR = Path("/data/logs/deterioration/allStay/allRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/mtand_m3care_similarityloss_w0_01_imputedMiss_maskFusion_tm8_et16_3_1ohe")
MTAND_MODALITIES = ("ecg", "cxr", "ts")

# Optional: copy the n values from the original plot/logs if desired.
# The saved heatmap CSVs do not include these counts.
MTAND_N_STAYS = {"ecg": 3644, "cxr": 789, "ts": 3347}  # e.g. {"ecg": 3644, "cxr": 789, "ts": 3347}

OUTPUT_DIR = "/data/outputs/report"

In [ ]:
def load_mtand_marginals(run_dir, modalities=("ecg", "cxr", "ts")):
    """Load saved attention heatmaps and reproduce the source script's marginal."""
    attention_dir = Path(run_dir) / "mtand_attention"
    if not attention_dir.is_dir():
        raise FileNotFoundError(f"Missing attention directory: {attention_dir}")

    series = {}
    for modality in modalities:
        path = attention_dir / f"attn_heatmap_{modality}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"Missing heatmap CSV: {path}")
        frame = pd.read_csv(path)
        obs_cols = [c for c in frame if re.fullmatch(r"obs_[-+]?\d+(?:\.\d+)?min", c)]
        if not obs_cols:
            raise ValueError(f"No observation-time columns in {path}")
        minutes = np.array([float(re.fullmatch(r"obs_([-+]?\d+(?:\.\d+)?)min", c).group(1)) for c in obs_cols])
        heat = frame[obs_cols].apply(pd.to_numeric, errors="raise").to_numpy(dtype=float)
        if not np.isfinite(heat).all() or (heat < 0).any():
            raise ValueError(f"Invalid attention values in {path}")
        marginal = heat.mean(axis=0)
        total = marginal.sum()
        marginal = marginal / total if total else marginal
        series[modality] = pd.Series(marginal, index=minutes, name=modality.upper())
    return pd.DataFrame(series).sort_index().rename_axis("minutes_since_ed_arrival")


def plot_mtand_marginal_from_csv(
    run_dir, modalities=("ecg", "cxr", "ts"), n_stays=None, run_name=None,
    figsize=(8, 5), save=False, filename="attn_marginal_over_time_replotted.png",
):
    """Plot the same mean-over-reference, normalised marginal as mtand_attention.py."""
    marginal = load_mtand_marginals(run_dir, modalities)
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)
    n_stays = n_stays or {}
    for modality in modalities:
        label = modality.upper()
        if modality in n_stays:
            label += f" (n={n_stays[modality]})"
        ax.plot(marginal.index, marginal[modality], marker="o", label=label)
    ax.set_xlabel("Observation time (minutes since ED arrival)")
    ax.set_ylabel("Normalised attention weight")
    if run_name:
        ax.set_title(f"Where mTAND attends over time (marginal over reference grid)\n{run_name}")
    else:
        ax.set_title(f"Where mTAND attends over time (marginal over reference grid)\n{Path(run_dir).name}")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    if save:
        destination = Path(OUTPUT_DIR) / "mtand_attention" / filename
        fig.savefig(
            destination,
            dpi=600,
            bbox_inches="tight",
            transparent=True,
        )
        print(f"Saved: {destination}")
    plt.show()
    return marginal, fig, ax


In [ ]:
mtand_marginal, fig, ax = plot_mtand_marginal_from_csv(
    mtand_m3care_path,
    run_name="M3Care + mTAND",
    modalities=MTAND_MODALITIES,
    n_stays=MTAND_N_STAYS,
    figsize=(6, 4),  
    save=True,     
    filename="m3care_attn_marginal_over_time.png"
)
display(mtand_marginal.round(4))


In [ ]:
mtand_marginal, fig, ax = plot_mtand_marginal_from_csv(
    mtand_fusemoe_path,
    run_name = "FuseMoE + mTAND",
    modalities=MTAND_MODALITIES,
    n_stays=MTAND_N_STAYS,
    figsize=(6, 4),
    save=True,     
    filename="fusemoe_attn_marginal_over_time.png"
)
display(mtand_marginal.round(4))
